In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
def load_dataset(file_path: str) -> pd.DataFrame:
    """Load the raw dataset"""
    if not os.path.exists(file_path):
        raise FileNotFoundError("Dataset not found")
    print(f"loading dataset from: {file_path}")
    return pd.read_csv(file_path)

In [6]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Data cleaning pipeline"""
    # Create a copy to avoid mutating the original dataset
    data = df.copy()
    
    # 1. Drop customerID first (or check for duplicates here if needed)
    id_col = ["customerID"]
    if "customerID" in data.columns:
        data.drop(columns=id_col, inplace=True)
        print(f"Dropped column: {id_col}")


    categorical_cols = data.select_dtypes(include=["object"]).columns
    for col in categorical_cols:
        data[col] = data[col].astype(str).str.strip().str.lower()
    
    
    # Categorical imputation (guarding against empty mode)
    for col in categorical_cols:
        if data[col].isnull().sum() > 0:
            mode_series = data[col].mode()
            if not mode_series.empty:
                data[col] = data[col].fillna(mode_series[0])
    
    # Numerical imputation
    num_cols = data.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if data[col].isnull().sum() > 0:
            median_val = data[col].median()
            data[col] = data[col].fillna(median_val)
            
    return data


In [4]:
def save_clean_data(df: pd.DataFrame, output_path: str):
    """Saves the processed Dataframe"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Successfully saved to: {output_path}\n")

In [7]:
CLEAN_DATA_PATH = "../data/cleaned_customer_churn.csv"
data = load_dataset("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
clean_df = clean_data(data)
save_clean_data(clean_df, CLEAN_DATA_PATH)

loading dataset from: ../data/WA_Fn-UseC_-Telco-Customer-Churn.csv
Dropped column: ['customerID']
Successfully saved to: ../data/cleaned_customer_churn.csv



C:\Users\sandi\AppData\Local\Temp\ipykernel_22388\2724809554.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = data.select_dtypes(include=["object"]).columns
